# 03 — Select questions

1. Drop questions whose video cannot be trusted (quality filter).
2. Keep only **Emotions** questions.
3. Put them in a fixed random order and save. The run notebook takes the first `N_QUESTIONS` of this order,
   so increasing N later only **adds** questions — earlier results stay valid.

Uses only `questions.parquet` and `videos.parquet` from notebook 01. No answer keys.

In [1]:
from pathlib import Path
import math
import pandas as pd

ROOT = Path.cwd().parent
DATA = ROOT / "data-private/processed"
questions = pd.read_parquet(DATA / "questions.parquet")
videos = pd.read_parquet(DATA / "videos.parquet").set_index("movie_title")
film = videos.loc[questions.movie_title].reset_index(drop=True)  # the film's row, aligned to each question
print(len(questions), "questions in", questions.movie_title.nunique(), "films")

2335 questions in 168 films


## 1. Quality filter

Only one question is dropped: **F5kB7** (film *STOP*). Its clip ends at 714.5 s, but the film is only 696.3 s long,
so part of the clip does not exist.

Nothing is dropped for alignment. The per-question check in notebook 01 looks at only ~30 s of audio, and it often picks
a wrong shift by chance. A film-level check using each film's full transcript (hundreds of words) found the best shift
at **0 s for 166 of 168 films**; the 2 exceptions have almost no speech (25 and 103 words), so the check is inconclusive there, not failed.

**Clips with fewer than 64 frames** (6 questions, 2–2.6 s long) are kept: they get all their frames instead of 64,
rounded down to an even number (Qwen needs frame pairs). The number used is stored in the `n_frames` column.

In [2]:
N_FRAMES = 64
past_end = questions.t_j > film.local_duration_s
print("dropped (clip ends after the video):", questions.loc[past_end, ["question_id", "movie_title"]].values.tolist())
good = questions[~past_end].copy()

# frames available in [t_i, t_j], counted the same way Qwen's reader does
film_good = film[~past_end.values]
available = (good.t_j * film_good.fps.values).apply(math.floor) - (good.t_i * film_good.fps.values).apply(math.ceil) + 1
good["n_frames"] = available.clip(upper=N_FRAMES) // 2 * 2
print("questions with fewer than", N_FRAMES, "frames:", (good.n_frames < N_FRAMES).sum(), good.loc[good.n_frames < N_FRAMES, "n_frames"].tolist())
print(f"kept {len(good)} of {len(questions)} questions, {good.movie_title.nunique()} films")

dropped (clip ends after the video): [['F5kB7', 'STOP']]
questions with fewer than 64 frames: 6 [48, 56, 62, 52, 62, 56]
kept 2334 of 2335 questions, 168 films


## 2. Emotions only

A question can have several categories; it is kept if *Emotions* is one of them.

In [3]:
emotion = good[good.cat_emotions]
print(len(emotion), "Emotions questions from", emotion.movie_title.nunique(), "films")
print("split:", emotion.split.value_counts().to_dict())
print("tagged with a visual cue (face/body):", emotion.cue_visual_any.sum())
print("no dialogue in the clip:", (emotion.focused_n_words == 0).sum())
print("clip length (s): median %.0f, max %.0f" % (emotion.focused_duration_s.median(), emotion.focused_duration_s.max()))

598 Emotions questions from 153 films
split: {'test': 510, 'validation': 88}
tagged with a visual cue (face/body): 506
no dialogue in the clip: 74
clip length (s): median 28, max 1161


## 3. Fixed random order, save

`order` 0, 1, 2, … is a shuffle with a fixed seed: the first N questions are a random sample, and the same N always gives the same questions.

In [4]:
SEED = 0
emotion = emotion.sample(frac=1, random_state=SEED).reset_index(drop=True)
emotion.insert(0, "order", range(len(emotion)))
emotion.to_parquet(DATA / "emotion_questions.parquet", index=False)
emotion[["order", "question_id", "split", "movie_title", "focused_duration_s", "cue_visual_any", "question"]].head(10)

,order,question_id,split,movie_title,focused_duration_s,cue_visual_any,question
0,0,FH-76,test,REELING ISAAC,21.260272,True,How does the man feel about knocking on the door?
1,1,97rJU,test,SWING FOR THE FENCES,47.547361,True,How does the young woman feel about the older ...
2,2,cP4lM,test,THE GUARD,293.952596,True,Why does the man refuse to use the knife at di...
3,3,XESWo,test,MOTHER,55.032380,True,How does the woman feel during the call?
4,4,6p7dK,test,GARFIELD,21.429371,True,How would one describe the expression on the m...
5,5,vXeRH,test,"MOTHER, CHILD",122.013046,True,Why did the man return despite being so determ...
6,6,IESLM,test,MARYAM JOON,71.187442,True,Why does the young woman touch the fruit to he...
7,7,aWgG2,validation,FATHER FIGURE,20.083487,True,How does the woman feel when she looks at her ...
8,8,xe0_s,validation,DOESN'T FALL FAR,27.535537,True,The two men share a look of _____ when the wom...
9,9,50aH_,test,ONE DAY NOTICE,36.950624,True,Why is the man in grey quietly mouthing the wo...
